In [1]:
import math,time
import torch
import torch.nn as nn
import torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print(device)

cuda


In [2]:
#softmax
def softmax(x):
  max_val=torch.max(x,dim=-1,keepdim=True).values
  e=torch.exp(x-max_val)
  return e/torch.sum(e,dim=-1,keepdim=True)

In [3]:
from torch._C import dtype
#注意力机制
def Scaled_Dot_Product_Attention(q,k,v,mask=None):
  #q:[B,H,t,dk]
  #k:[B,H,t,dk]
  #v:[B,H,t,dv]
  dk=q.size(-1)
  scores=q@k.transpose(-2,-1)/math.sqrt(dk)
  if mask is not None:
    scores=scores.masked_fill(~mask,float("-inf"))
  attn=softmax(scores)
  return attn@v,attn

In [4]:
#多头注意力机制
class MutiHeadAttention(nn.Module):
  def __init__(self,d_model,n_head,dropout=0.01)->None:
    super().__init__()
    assert d_model%n_head==0
    self.d_model=d_model
    self.h=n_head
    self.dk=d_model//n_head
    self.dropout=nn.Dropout(dropout)
    self.W_q=nn.Linear(d_model,d_model)
    self.W_k=nn.Linear(d_model,d_model)
    self.W_v=nn.Linear(d_model,d_model)
    self.W_o=nn.Linear(d_model,d_model)

  def split(self,x):
    B,T,_=x.shape
    return x.view(B,T,self.h,self.dk).transpose(1,2)

  def forward(self,q,k,v,mask=None):
    q=self.split(self.W_q(q))
    k=self.split(self.W_k(k))
    v=self.split(self.W_v(v))
    out,self.last_attn=Scaled_Dot_Product_Attention(q,k,v,mask)
    out=self.dropout(out)
    B,h,T,dk=out.shape
    return self.W_o(out.transpose(1,2).contiguous().view(B,T,h*dk))

mha=MutiHeadAttention(64,8)
q,k,v=torch.randn(2,8,64),torch.randn(2,8,64),torch.randn(2,8,64)
causal=torch.tril(torch.ones(8,8,dtype=torch.bool))[None,None]
out=mha(q,k,v,causal)
print(out)




tensor([[[-0.7192, -0.0225, -0.3360,  ...,  0.0461, -0.1789, -0.1847],
         [ 0.1984,  0.0753,  0.0518,  ...,  0.1990, -0.2845, -0.3615],
         [ 0.1296,  0.0136,  0.1238,  ..., -0.0898, -0.2083, -0.2430],
         ...,
         [ 0.0651,  0.0679, -0.1045,  ..., -0.0078, -0.1156, -0.1715],
         [ 0.0660,  0.1727,  0.0038,  ..., -0.1187, -0.1668, -0.1608],
         [ 0.1009,  0.1916,  0.0145,  ..., -0.0023, -0.1692, -0.1267]],

        [[ 0.1441,  0.4987, -0.2703,  ...,  0.7179,  0.1101, -0.0281],
         [ 0.3629,  0.7401, -0.0219,  ...,  0.2095,  0.2612, -0.2792],
         [ 0.2094,  0.2984, -0.1426,  ...,  0.2544,  0.0471, -0.1194],
         ...,
         [ 0.2027,  0.2326,  0.1375,  ...,  0.1232, -0.0552, -0.1920],
         [ 0.0554,  0.2558,  0.1715,  ...,  0.0813, -0.0441, -0.2061],
         [ 0.1777,  0.1966,  0.1168,  ...,  0.0876, -0.0312, -0.2142]]],
       grad_fn=<ViewBackward0>)


In [5]:
#FFN
class FeedForwardNet(nn.Module):
  def __init__(self,d_model,d_dff,dropout=0.01)->None:
    super().__init__()
    self.net=nn.Sequential(nn.Linear(d_model,d_dff),nn.GELU(),nn.Linear(d_dff,d_model),nn.Dropout(dropout))
  def forward(self,x):return self.net(x)

In [6]:
#位置编码
class SinsoidialPE(nn.Module):
  def __init__(self,d_model,max_len=2048)->None:
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    pos=torch.arange(0,max_len).unsqueeze(1).float()
    div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(pos*div)
    pe[:,1::2]=torch.cos(pos*div)
    self.register_buffer("pe",pe)

  def forward(self,x):
    return x+self.pe[:x.size(1)]

In [7]:
#encoderlayer
class EncoderLayer(nn.Module):
  def __init__(self,d_model,n_head,d_dff,dropout=0.01)->None:
    super().__init__()
    self.ln1,self.ln2=nn.LayerNorm(d_model),nn.LayerNorm(d_model)
    self.ffn=FeedForwardNet(d_model,d_dff,dropout)
    self.attn=MutiHeadAttention(d_model,n_head,dropout)
    self.dropout=nn.Dropout(dropout)
  def forward(self,x,mask=None):
    x=x+self.dropout(self.attn(self.ln1(x),self.ln1(x),self.ln1(x),mask))
    return x+self.dropout(self.ffn(self.ln2(x)))

In [8]:
#decoderlayer
class DecoderLayer(nn.Module):
  def __init__(self,d_model,d_dff,n_head,dropout=0.01)->None:
    super().__init__()
    self.attn=MutiHeadAttention(d_model,n_head,dropout)
    self.cross_attn=MutiHeadAttention(d_model,n_head,dropout)
    self.ffn=FeedForwardNet(d_model,d_dff,dropout)
    self.ln1,self.ln2,self.ln3=nn.LayerNorm(d_model),nn.LayerNorm(d_model),nn.LayerNorm(d_model)
    self.dropout=nn.Dropout(dropout)
  def forward(self,x,memory,src_mask,tgt_mask):
    x=x+self.dropout(self.attn(self.ln1(x),self.ln1(x),self.ln1(x),tgt_mask))
    x=x+self.dropout(self.cross_attn(self.ln2(x),memory,memory,src_mask))
    return x+self.dropout(self.ffn(self.ln3(x)))


In [9]:
#transformer
class Transformer(nn.Module):
  def __init__(self,src_voc,tgt_voc,max_len=2048,pad_idx=0,d_model=128,n_head=4,d_dff=512,n_layers=4,dropout=0.01)->None:
    super().__init__()
    self.padIdx,self.d_model=pad_idx,d_model
    self.enc_ln,self.dec_ln=nn.LayerNorm(d_model),nn.LayerNorm(d_model)
    self.pe=SinsoidialPE(d_model,max_len)
    self.enc=nn.ModuleList(EncoderLayer(d_model,n_head,d_dff,dropout) for _ in range(n_layers))
    self.dec=nn.ModuleList(DecoderLayer(d_model,d_dff,n_head,dropout) for _ in range(n_layers))
    self.dropout=nn.Dropout(dropout)
    self.src_emb=nn.Embedding(src_voc,d_model,padding_idx=pad_idx)
    self.tgt_emb=nn.Embedding(tgt_voc,d_model,padding_idx=pad_idx)
    self.out=nn.Linear(d_model,tgt_voc)

  def make_mask(self,src,tgt):
    src_mask=(src!=self.padIdx)[:,None,None,:]
    Tt=tgt.size(1)
    causal=torch.tril(torch.ones(Tt,Tt,dtype=torch.bool,device=tgt.device))[None,None]
    tgt_mask=(tgt!=self.padIdx)[:,None,None,:]&causal
    return src_mask,tgt_mask

  def encode(self,src,src_mask):
    x=self.dropout(self.pe(self.src_emb(src)*math.sqrt(self.d_model)))
    for layer in self.enc:x=layer(x,src_mask)
    return self.enc_ln(x)

  def decode(self,src,memory,src_mask,tgt_mask):
    x=self.dropout(self.pe(self.tgt_emb(src)*math.sqrt(self.d_model)))
    for layer in self.dec:x=layer(x,memory,src_mask,tgt_mask)
    return self.out(self.dec_ln(x))
  def forward(self,src,tgt):
    src_mask,tgt_mask=self.make_mask(src,tgt)
    return self.decode(tgt,self.encode(src,src_mask),src_mask,tgt_mask)
model = Transformer(20, 20)
src = torch.randint(3, 20, (2, 8)); tgt = torch.randint(3, 20, (2, 9))
print(model(src, tgt).shape, "params:", sum(p.numel() for p in model.parameters()))





torch.Size([2, 9, 20]) params: 1859604


In [10]:
#学习序列反转
PAD, BOS, EOS, V = 0, 1, 2, 20

def make_batch(B=64,L=8):
  src=torch.randint(3,V,(B,L))
  rev=src.flip(1)
  tgt_in=torch.cat([torch.full((B,1),BOS),rev],1)
  tgt_out=torch.cat([rev,torch.full((B,1),EOS)],1)
  return src.to(device), tgt_in.to(device), tgt_out.to(device)


model=Transformer(V,V).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=5e-4,)
steps=1000

for step in range(steps):
  src,tgt_in,tgt_out=make_batch()
  logits=model(src,tgt_in)
  loss=F.cross_entropy(logits.reshape(-1,V),tgt_out.reshape(-1),ignore_index=PAD)
  opt.zero_grad();loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
  opt.step()
  if step % 100 == 0: print(step, round(loss.item(), 4))

0 3.1924
100 1.9329
200 1.6124
300 0.9329
400 0.7084
500 0.6683
600 0.5126
700 0.4211
800 0.3729
900 0.3629


In [17]:
@torch.no_grad()
def greedy_decoder(model,src,max_len=200):
  model.eval()
  src_mask=(src!=PAD)[:,None,None,:]
  memory=model.encode(src,src_mask)
  ys=torch.full((src.size(0),1),BOS,device=src.device)
  for _ in range(max_len):
    _,tgt_mask=model.make_mask(src,ys)
    logits=model.decode(ys,memory,src_mask,tgt_mask)
    nxt=logits[:,-1].argmax(-1,keepdim=True)
    ys=torch.cat([ys,nxt],1)
    if (nxt==EOS).all():break
  return ys[:,1:ys.size(1)-1]
src,_,tgt_out=make_batch(5)
pred=greedy_decoder(model,src,max_len=9)
for s, p in zip(src.tolist(), pred.tolist()): print(s, "->", p)
print("exact match acc:", (pred == tgt_out[:, :8]).all(1).float().mean().item())

[11, 8, 16, 4, 5, 10, 8, 15] -> [15, 8, 10, 5, 4, 16, 8, 11]
[4, 14, 16, 19, 7, 9, 18, 7] -> [7, 18, 9, 7, 19, 16, 14, 4]
[11, 12, 19, 11, 17, 4, 6, 5] -> [5, 6, 4, 17, 11, 19, 12, 11]
[5, 18, 13, 4, 10, 19, 18, 18] -> [18, 18, 19, 10, 4, 13, 18, 5]
[9, 13, 17, 15, 19, 10, 3, 6] -> [6, 3, 10, 19, 15, 17, 13, 9]
exact match acc: 1.0
